# 01 — Document ingestion and parsing: Steps 1–8

This notebook implements only **source loading, DOCX inventory, parser setup, ordered structural block extraction, conservative document/region/marker detection, assembly of the intermediate corpus representation, metadata provenance classification, and validated Stage 1 persistence** from [the approved plan](../docs/stage-01-ingestion-parsing-plan.md).

Run from top to bottom using the project's `.venv` kernel. Setup from the project directory: `python -m pip install -e '.[dev]'`.

We read the combined DOCX directly from its archive, inspect the stored structure, compare a **small sample** with `python-docx`, then extract ordered structural records in Step 4 add separate evidence-backed annotations in Step 5, assemble them into one corpus record in Step 6, distinguish metadata origins in Step 7, and validate/persist the result in Step 8. Steps 1–7 stay in memory; Step 8 writes only the processed JSON artifact after validation passes. Reference chunks and evaluation data are not read.

Each section explains what, why, how, and what to inspect. Preview truncation affects display only. This is structural inspection, not cleaning or a complete parsing pipeline.
Implementation lives in `src/property_copilot/ingestion/`. Run cells in order;
inspect results and call reusable helpers through `workflow`.


## Step 1 — Locate and load the exact source

**What/why:** Select the exact combined DOCX so reference chunks and companion files cannot accidentally become input.

**How:** Resolve the project root from either the project or notebook directory. Read only the named ZIP member into `BytesIO`; do not extract the archive. Hash the same bytes that the parser will see.

**Choice:** In-memory loading avoids an extra copy on disk. Extraction would help manual Word viewing but is unnecessary here. A hash identifies this file snapshot, not a legal instrument. Missing, duplicate, or corrupt source entries should fail visibly.

In [ ]:
from property_copilot.ingestion import IngestionWorkflow

workflow = IngestionWorkflow()

workflow.step_1_locate_and_load_the_exact_source()


**Inspect:** The member must end in `full_text/ALL_FULL_TEXTS.docx`; the inspected snapshot is 940,203 bytes. No timestamp is manufactured. The SHA-256 enables comparison on later reruns.

## Step 2 — Inventory the DOCX package

**What/why:** A DOCX is itself a ZIP containing XML parts. Inspect these before relying on a higher-level parser, which may not expose every Word feature.

**How:** Read document XML and style definitions with the standard library. Inventory all package names but do not load unrelated archive members. XML namespaces distinguish Word elements from other formats.

**Choice:** XML is explicit but verbose. We use it for counts and representative inspection, not to build a custom full parser.

In [ ]:
workflow.step_1_locate_and_load_the_exact_source_2()


In [ ]:
workflow.step_1_locate_and_load_the_exact_source_3()


**Interpretation:** 22 Heading 1 paragraphs include the corpus title and 21 source titles. One Word section is a layout setting, not 21 legal-document boundaries. The 252 cell paragraphs account for the difference between 31,055 top-level and 31,307 total paragraphs.

A style being defined in `styles.xml` does not mean it is used. Parts and chapters may both have Heading 2, so style depth alone cannot establish legal hierarchy. The 41 explicit page breaks do not tell us the rendered page count.

In [ ]:
workflow.step_1_locate_and_load_the_exact_source_4()


**Inspect:** The metadata tables contain jurisdiction, URL, official reference, version date, access date, and notes. These are source claims, not verified legal facts. The full-text and reference-rule markers are distinct. Bulgarian `Чл.` provisions can be ordinary paragraphs while EU `Член` provisions can be headings.

Reference rules remain in the source untouched; their later labeling is Step 5. Here we only view examples. No metadata mapping or region assignment is implemented.

In [ ]:
workflow.step_1_locate_and_load_the_exact_source_5()


**Interpretation:** The footer's `PAGE` instruction is a dynamic Word field; it has no usable rendered page value here. We will not invent page numbers. The scan highlights common omission risks and unknown body elements; it is not a universal DOCX validator.

## Step 3 — Set up the parser and compare a small region

**What/why:** Use `python-docx` for readable paragraph, style, run, and table objects, with XML inspection as an independent check. Its public `iter_inner_content()` preserves paragraph/table order. Some features, including paragraphs inside revision marks, need additional handling; see the [official API](https://python-docx.readthedocs.io/en/latest/api/document.html).

**How:** Open the same bytes and inspect only the first document's title, subtitle, table, and next heading. The dependency is `python-docx>=1.2,<2`; versions are printed below. The range allows compatible updates, while the recorded versions identify this run.

**Alternatives/trade-offs:** XML-only parsing offers more control but requires more Word-specific code. [docx2python](https://github.com/ShayHill/docx2python) supports additional content types but adds a nested representation. [Mammoth](https://github.com/mwilliamson/python-mammoth) provides readable HTML while intentionally discarding some formatting detail. Plain text loses the structure we need. We introduce no RAG framework.

**Inspect:** Compare text and style IDs (not localized display names), table dimensions, and every cell of this one metadata table. This is a bounded demonstration, not Step 4's corpus-wide extraction.

An absent XML paragraph style means “use the default,” not “no style.” We read the default paragraph style from `styles.xml` and compare it with the resolved style returned by `python-docx`. The inventory still distinguishes explicit style assignments from defaults.

In [ ]:
workflow.step_1_locate_and_load_the_exact_source_6()


## Step 4 — Extract ordered structural blocks

**What:** Create `ordered_blocks`: one record for each body paragraph or table, in source order. Tables contain ordered rows, cells, and paragraph records. `auxiliary_content` holds footer/header records separately; `layout_markers` accounts for Word section properties.

**Why:** Separate paragraph and table lists lose their interleaving. A single text string loses formatting, empty paragraphs, and table coordinates. We preserve these structures so you can inspect them before deciding any later processing.

**How:** Pair `python-docx.iter_inner_content()` with the independent XML inventory. The public API supplies paragraph/run text and formatting; XML supplies exact structural positions and explicit break/field events. Every body element must be accounted for, or extraction stops with a location-specific error.

**Scope:** These are Step 4 block records, not Step 6's complete corpus/document representation. No document IDs, regions, legal roles, inferred metadata, or chunk relationships are assigned. Nothing is exported.

### 4a. Record conventions and source locations

| Field | Meaning |
|---|---|
| `id` | SHA-256 of source-file hash, XML part, and structural path |
| `position` | Zero-based child position within its immediate container |
| `source` | XML part and zero-based structural path; this is a locator convention, not XPath |
| `raw_text` | Paragraph text returned by the parser, without trimming or normalization |
| `style` | Explicit style ID and resolved default ID/name; no semantic interpretation |
| `runs` | Ordered run text, direct formatting, and ordered XML content events |
| `break_markers` | Break type and its run/event position, including invisible page breaks |
| `rows` | For tables: ordered cells containing their paragraph records |

A cell paragraph location looks like `word/document.xml` + `/body/children/26/rows/0/cells/1/children/1`. IDs are reproducible for identical source bytes and locations, not permanent legal identifiers across revisions.

**Text choice:** Text runs keep spaces and Unicode exactly. `raw_text` follows Word parser semantics: a tab is `\t`, a line break is `\n`, and a page/column break or field instruction has no visible character. The ordered events preserve those distinctions; no paragraph separators are inserted. A table has no canonical flattened text.

**Formatting choice:** `None` in bold/italic/etc. means “not directly specified,” not false. We retain direct formatting and serialized property XML instead of computing Word's full inherited appearance. Serialized XML preserves element information, not byte-for-byte XML formatting.

In [ ]:
workflow.step_4_extract_ordered_structural_blocks()


### 4b. Paragraphs, runs, and explicit breaks

**What/how:** Keep each paragraph even when its visible text is empty. Pair runs with their XML siblings and preserve event order within each run. Record bold, italic, underline, strike, superscript/subscript, font name, and size; full direct run properties retain additional details such as color.

**Trade-off:** This extractor targets the inspected source's direct paragraph/run structure. It fails on unexpected paragraph containers instead of silently omitting tracked changes or content controls. Unhandled run elements remain visible as XML events and are counted in the checks below. This is not a universal Word renderer.

**Inspect next:** A page-break-only paragraph should have empty `raw_text` but a nonempty `break_markers` list. No filtering of empty strings is permitted.

In [ ]:
# Helpers for this step are imported by the package workflow.


### 4c. Tables remain in place, with nested cell paragraphs

**What/how:** Store each table at its original body position. Iterate rows and cells once; each cell holds its own ordered paragraph records using the same paragraph function.

**Choice:** Preserve the six-by-two tables as cells, not dictionaries of inferred metadata. Coordinates express structure only. The observed tables are rectangular, unmerged, and unnested. Explicit checks reject merged, nested, or omitted-cell layouts requiring another representation, rather than duplicating or misplacing their content. General support for those layouts is unnecessary for this source.

**Inspect next:** A title paragraph, subtitle paragraph, table, and following paragraph must appear in that order. The first table's cell text is shown with row and column coordinates.

In [ ]:
workflow.step_4c_tables_remain_in_place_with_nested_cell_paragraphs()


### 4d. Footer content stays outside the body

**What/why:** Preserve existing footer/header parts once each. Repeating a footer inside body text would change reading order and duplicate content.

**How:** Read existing package parts and wrap their existing paragraph/table elements with `python-docx`. This small lower-level adapter does not create headers or footers. The same record functions preserve the footer's empty visible text and ordered `PAGE` field events.

**Trade-off:** Word evaluates fields when rendering pages. We keep the field instruction and do not calculate a page number. Section-property XML remains in `layout_markers`; no legal document boundaries are derived from it.

In [ ]:
workflow.step_4d_footer_content_stays_outside_the_body()


### 4e. Inspect the records

The summary below uses short previews only; the underlying `raw_text` is complete. Table character counts sum cell paragraphs without adding artificial separators. The expandable JSON viewer shows a selected record in full, including source location, run formatting, and events. Change `SELECTED_BODY_POSITION` and rerun the viewer cell to inspect another record.

Look for the table at body position 26, the retained page-break-only paragraph, and the footer's field instruction. Table-cell paragraph positions count original XML children, including the preceding `tcPr` property element; they are source coordinates, not a count of just paragraphs.

In [ ]:
workflow.step_4e_inspect_the_records()


### 4f. Check Step 4 preservation

**What/why:** Counts alone can pass even if text or ordering changes. These focused checks compare every body/cell paragraph with independent XML, verify identifiers and source positions, and reconcile all explicit breaks. They validate this extraction step; they do not perform the later document-level validation/export stage.

**How:** An independent XML text reader applies only Word's text-encoding conventions for tabs and line breaks. Page/column breaks and field instructions are checked as separate events. We check literal text fragments as well as the visible paragraph string, so invisible events or spaces cannot silently disappear.

A small in-memory Word fixture additionally checks an interleaved paragraph/table/paragraph sequence, multiple cell paragraphs, whitespace, formatting, and a page break between two text fragments. It is never written to disk. This tests behavior that the source's regular tables and page-break-only paragraphs do not cover.

In [ ]:
workflow.step_4f_check_step_4_preservation()


In [ ]:
workflow.step_4f_check_step_4_preservation_2()


## Step 5 — Identify document boundaries and structural markers

**What:** Use Step 4's `ordered_blocks` to identify source-document spans, distinguish corpus front matter, assign structural regions, and collect candidate legal markers.

**Why:** `Heading1` alone cannot distinguish the corpus title from a source title. Reference-rule excerpts also contain article citations, so text patterns alone are insufficient.

**How:** Create separate annotation dictionaries keyed by existing block IDs. `ordered_blocks`, nested cell paragraphs, footer records, and original styles remain unchanged. A before/after checksum checks that no Step 4 record was modified.

**Boundary evidence:** A nonempty `Heading1` paragraph immediately followed by a non-heading subtitle paragraph and the observed six-row/two-column table labels. We inspect table labels only to recognize the structure; no jurisdiction/date/URL values are interpreted. The corpus heading lacks this pattern.

**Conservative choice:** No fuzzy title matching, page-break-based boundaries, or hardcoded split positions. Unmatched Heading 1 paragraphs after the first verified source raise an explicit review error. A valid source span must contain exactly one exact Heading 2 full-text marker and at most one preceding reference-rules marker. Unexpected layouts or ambiguous markers stop detection instead of being guessed.

**Representation:** `document_boundaries` holds minimal source spans and evidence IDs. `block_annotations` holds membership, region, and observed Word heading level. These are Step 5 annotations, not Step 6's combined corpus representation or a legal hierarchy. Span indices are zero-based indices into `ordered_blocks`; ends are exclusive. Displayed body positions use Step 4's source coordinates.

### 5a. Recognize and audit document starts

Inspect the full Heading 1 audit below. There should be 22 candidates: the corpus title plus 21 verified source starts. IDs are anchored to the source title block, not inferred legal-document identifiers.

In [ ]:
workflow.step_5_identify_document_boundaries_and_structural_markers()


### 5b. Assign regions using explicit delimiters

| Region | Rule |
|---|---|
| `front_matter` | All blocks before the first verified source title; document ID is `None` |
| `document_metadata` | Source title, subtitle, table, and any following blocks before a region marker |
| `reference_rules` | Exact reference-rules heading through the block before the full-text heading, when present |
| `full_text` | Exact full-text heading through the block before the next verified source title, or end of body |

**Choices:** Each region marker belongs to the region it introduces. A separator/page-break paragraph before the next title remains in the preceding span, unchanged; breaks alone do not change membership. The two documents without reference-rule sections get no invented reference region. Table-cell paragraphs inherit their containing table's membership and region. Footer and layout records remain separate.

Exact headings are required so a sentence that merely quotes “Full text” cannot change the region. Duplicate, missing, or reversed markers raise a review error. Labels identify the corpus structure; they do not establish legal authority, currency, or source reliability.

**Inspect:** The 21-row inventory shows inclusive body positions, region counts, and full-text evidence positions. The region summary includes every body block exactly once.

In [ ]:
workflow.step_5b_assign_regions_using_explicit_delimiters()


### 5c. Review complete spans and boundary evidence

**What/how:** First inspect a small window around every verified start, including the preceding block. Then choose a document to view its complete boundary descriptor, all region intervals, the original opening text, and the ending/next-source transition.

The default is source 20, a short document without reference rules. Its expandable block listing includes **every block and complete text** in the span, not just an opening sample. Choose source 1–21 to inspect another document. Descriptors point to original blocks rather than duplicating or rewriting them.

**Inspect:** The corpus title and numbered source list stay in `front_matter`. For the selected source, check that its title/subtitle/table start the metadata region, the full-text heading changes the region once, and the following document title lies outside its exclusive end. Shortened text in the all-boundaries table is display-only.

In [ ]:
workflow.step_5c_review_complete_spans_and_boundary_evidence()


### 5d. Identify candidate structural/legal markers

**What/how:** Run anchored patterns on full-text paragraphs only. Candidate annotations store the original block ID, source location, exact matched label, character span, rule, and observed Word heading level. Annotations never split paragraphs, merge headings with following titles, or create article spans.

**Conservative rules:**

- Part (`Част`), Chapter (`Глава`), Section (`Раздел`), and Annex (`Приложение`) require both a leading label and an explicit Word heading level 2 or 3. The following label token is retained as written; no numbering conversion is attempted.
- Article (`Чл.` / `Член`) and `§` require a leading numeric identifier, with an optional Cyrillic suffix such as `2а`. They may occur in ordinary paragraphs. Inline mentions and `Членове` are not provision starts.
- Patterns ignore letter case only during matching and tolerate leading spaces/tabs; stored strings are never lowercased, stripped, normalized, or rewritten.
- Reference rules, metadata, front matter, and footers receive no legal-marker candidates. All remain preserved. Exact region-marker headings are delimiters, not legal candidates.

**Trade-off:** Favor inspectable evidence over broad recall. A leading article citation can still be quoted text; it remains a **candidate**, not a confirmed provision. An unstyled chapter label is left unclassified for review. This deliberately avoids guessing a legal hierarchy from Word styles.

**Inspect:** Examples cover every requested marker, both article forms, and repeated/ambiguous headings. Headings with no matching rule remain visible in a review list; they are not discarded.

In [ ]:
workflow.step_5d_identify_candidate_structural_legal_markers()


### 5e. Check coverage, conservative behavior, and unchanged records

**What/why:** Verify the detected spans partition the body after front matter, each block has exactly one region, and candidate evidence points to unchanged source text. A checksum checks all Step 4 body records, including nested paragraphs and formatting, rather than just preview strings.

**Small focused cases:** A source may omit reference rules; missing/duplicate/out-of-order markers and an unmatched internal Heading 1 must fail. `Чл. 2а.` and `Член 1` should match at the start; inline citations, “Членове,” unstyled chapter labels, and reference-rule excerpts should not. Tests use small copied records in memory and do not alter the source.

These checks concern structural detection only. They do not verify legal meaning or establish a final hierarchy.

In [ ]:
workflow.step_5e_check_coverage_conservative_behavior_and_unchanged_records()


In [ ]:
workflow.step_5e_check_coverage_conservative_behavior_and_unchanged_records_2()


## Step 6 — Assemble the simple intermediate representation

**What:** Build a plain Python `corpus` dictionary from the already-approved Step 4 records and Step 5 annotations. It contains provenance, 21 document descriptors, ordered blocks, auxiliary content, layout markers, and diagnostics.

**Why:** One inspectable representation lets later experiments reuse the same source structure. It does not depend on a chunking strategy, database, or RAG framework.

**How:** Deep-copy the existing records, attach their existing membership and candidate-marker annotations, and point document descriptors to ranges in the ordered block list. No new boundary or marker detection is performed here.

| Component | Contents |
|---|---|
| `schema_version` | Version 1 of this notebook's intermediate record format |
| `source` | Existing archive/member path, DOCX byte size, SHA-256 |
| `parser` | Parser and XML-inspection implementation/version information |
| `documents` | Existing IDs, source title/subtitle, block range, raw metadata-table rows and evidence |
| `blocks` | Ordered Step 4 records plus Step 5 document, region and structural annotations |
| `auxiliary_content` | Existing footer/header records, including field instructions |
| `layout_markers` | Existing Word section-property records |
| `diagnostics` | Existing inventory/region counts, limitations and assembly-check outcomes |

**Design choices:** The body text exists in one ordered block list, with table paragraphs nested in their original cells. Descriptors reference ranges; they do not contain another full copy of each document. Small title/metadata values are repeated for inspection and retain exact block/cell evidence. Deep copies keep the assembled record independent of earlier notebook variables; the extra memory is acceptable for this source. No production classes or package refactor are introduced.

**Scope boundary:** Carry source metadata rows as verbatim labels and values, not normalized fields such as an interpreted effective date. Step 7's provenance classification and metadata interpretation are deferred. UTF-8 JSON is checked in memory; saving/reloading a parsed artifact belongs to Step 8 and is not implemented.

### 6a. Merge existing records and annotations without changing the inputs

Each body/cell paragraph retains its Step 4 ID, source location, raw text, style, runs and break events. `document_id` and `region` come directly from Step 5. `structural_annotations` carries the existing Word heading level, evidence ID, and candidate marker, if any. Front matter keeps `document_id=None`. Footer content stays separate and is not assigned to a source document.

**Inspect later:** The record viewer will show a full paragraph with these additions alongside its original text and formatting.

In [ ]:
workflow.step_6_assemble_the_simple_intermediate_representation()


### 6b. Build complete document descriptors with source evidence

**What/how:** Copy each verified boundary's ID, ordinal and evidence. Include title and subtitle exactly as written, and retain the zero-based `[start_index, end_index_exclusive)` range into `corpus['blocks']`. Source body positions remain separate from list indices.

For metadata, each table row keeps its row/column coordinates and separate label/value paragraph lists. Each paragraph includes its exact text, ID and source locator. This avoids joining paragraphs, changing date formats, interpreting jurisdiction strings, or mapping labels to a new metadata vocabulary.

**Trade-off:** Raw rows are less convenient for querying than typed metadata, but they preserve what the source actually supplied and leave Step 7's decisions open. Missing reference-rule sections remain `None` in the existing evidence; nothing is inferred.

**Inspect:** The first document descriptor is displayed in full below, including all six metadata rows and their evidence.

In [ ]:
workflow.step_6b_build_complete_document_descriptors_with_source_evidence()


### 6c. Assemble the corpus envelope

**What/how:** Combine the documents and blocks with the existing source manifest, exact installed parser versions, auxiliary/layout records and diagnostic counts. No timestamp is invented and no package metadata or external source is newly read.

The diagnostic notes describe limitations already visible in earlier steps: candidate markers remain provisional, some Word headings are unclassified, and a footer field does not supply rendered page numbers. They do not imply that legal content has been verified.

**Inspect:** The overview shows keys and collection sizes, not a dump of the whole corpus. A document descriptor references `blocks[start:end]`; table cells remain nested under a table in that sequence.

In [ ]:
workflow.step_6c_assemble_the_corpus_envelope()


### 6d. Inspect representative assembled records

The first display is one **complete document descriptor**, including raw metadata and every evidence reference. Expand the table and paragraph displays to see full records rather than previews.

The contiguous sequence shows the title, subtitle, metadata table, and following reference-region heading. Their shared document ID connects them to the descriptor; their regions retain the Step 5 distinctions. An article paragraph demonstrates how a candidate marker attaches to a complete original paragraph without splitting it. A separate footer view shows its field events.

Only the short sequence table truncates displayed text. The expandable JSON records contain the complete selected values.

In [ ]:
workflow.step_6d_inspect_representative_assembled_records()


### 6e. Check assembly integrity and JSON compatibility

**What/why:** Check that assembly only adds the approved annotations: IDs/order, every original field, document ranges, metadata evidence, candidate markers and auxiliary content must still agree with Steps 4–5. These are local assembly checks, not RAG evaluation or Step 8's full parsing validation/export stage.

**How:** Compare original fields to the assembled copies and resolve every descriptor's evidence references. Check the earlier inputs' digests again. Finally encode the corpus with `json.dumps(..., ensure_ascii=False, allow_nan=False)` as UTF-8 **in memory**, displaying only the byte count. No JSON file is written and no saved-artifact round trip is performed.

**Choice:** Ordinary dictionaries/lists are sufficient at this learning stage. The JSON check catches unsupported Python objects while preserving readable Cyrillic. Version 1 describes this intermediate interface, not a finalized legal-data schema.

In [ ]:
workflow.step_6e_check_assembly_integrity_and_json_compatibility()


## Step 7 — Separate extracted, generated, structurally interpreted, and deferred metadata

**What:** Add an in-memory `corpus['metadata_provenance']` catalog that classifies existing metadata without replacing Step 6's source records. The catalog covers all 21 documents; the example uses the **Black Sea Coast Spatial Development Act**.

**Why:** A source's “Version date” string is not a legal effective-date conclusion. A source heading is different from our candidate interpretation of it. A file hash or record ID is generated by ingestion, even though it helps locate source evidence.

| Category | Meaning in this notebook |
|---|---|
| `source_extracted` | Verbatim source titles, metadata-table labels/values, explicit marker text/styles, and separately scoped DOCX package properties |
| `generated` | File hash, measured byte size, local archive/member locator, IDs, encounter ordinal, positions/counts, parser/version and schema version |
| `structurally_interpreted` | Existing detected boundaries, membership, region assignments, resolved heading levels and candidate legal-marker roles |
| `deferred` | Decisions/fields not computed: normalized legal hierarchy, legal-date interpretation, language/topic/property inference, chunk metadata and its inheritance policy |

**How:** Build categorized views from the assembled `corpus`. Add friendly field keys by exact bilingual label lookup; preserve each original label and every value paragraph, with its table/cell/paragraph evidence. No values are stripped, normalized, translated, parsed as dates, repaired, reconciled, or filled in. Unknown or duplicate source rows would remain separate entries, not be overwritten.

**Important distinctions:** The container name `corpus['source']` means ingestion provenance; its local path/hash/byte count are not claims extracted from a legal source. A source URL inside a document's table *is* source-extracted metadata. Evidence locations are code-generated addresses pointing to unchanged source values.

The catalog is additive. Existing Step 6 fields stay unchanged. No metadata is copied into chunks, and **which document fields chunks should inherit remains undecided**. Existing table-cell membership is structural bookkeeping, not a chunk inheritance policy.

### 7a. Preserve raw metadata fields with evidence

**What/how:** Carry all six table fields — Jurisdiction, Source URL, Official Reference, Version Date, Accessed, Note — and titles into `source_extracted`. Friendly keys such as `version_date` classify labels only; the raw date strings remain authoritative source values for this stage. Lists preserve paragraph boundaries rather than joining them into new text.

**Trade-off:** Strings and evidence lists are less convenient than normalized typed fields, but avoid conflating the source's claims with our interpretation. Missing/empty values remain as supplied. Legal currency and validity are not verified here.

In [ ]:
workflow.step_7_separate_extracted_generated_structurally_interpreted_and_deferred_me()


### 7b. Keep package properties separate from legal-document metadata

**What/how:** Read `docProps/core.xml` from the same in-memory DOCX bytes loaded in Step 1. Preserve every property's exact text value, qualified XML name, attributes, and element location. Empty elements retain `None` as their XML text value; they are not filled from other documents.

**Why:** The package creator is `python-docx`, and its created/modified strings are `2013-12-23T23:15:00Z`. These values were already in the supplied DOCX. They are **source-extracted package properties**, not timestamps generated by our ingestion, not legal publication/version/effective dates, and not evidence of current legal validity.

No source timestamp is copied into an ingestion timestamp. No current ingestion timestamp is invented. No effort is made to reproduce reference-chunk dates or hours.

**Inspect:** The package-property table below is corpus-scoped; these fields are not attached to each legal document or any future chunks.

In [ ]:
workflow.step_7b_keep_package_properties_separate_from_legal_document_metadata()


### 7c. Make generated, interpreted, and deferred fields explicit

**Generated:** Record IDs, positions and counts are computed, even when computed deterministically. The ordinal is the parser's encounter order. The measured file path/bytes/hash and software versions describe the ingestion run, not the legal source. No generated timestamp is required.

**Structurally interpreted:** Choosing which blocks belong to a document or region, resolving Word heading levels, and identifying a candidate article/annex are code decisions with source evidence. Candidate roles remain provisional. The original heading text/style is source-extracted; its interpreted role is a separate field.

**Deferred:** List future fields as *not computed*, without placeholder values that could be mistaken for extracted facts. Do not decide chunk inheritance, which metadata belongs on chunks, or how parent/child chunks should propagate metadata.

The field guide maps existing corpus paths to their provenance categories. It describes existing fields; it neither rewrites them nor assigns a final legal hierarchy.

In [ ]:
workflow.step_7c_make_generated_interpreted_and_deferred_fields_explicit()


### 7d. Actual document example — Black Sea Coast Spatial Development Act

**Ordinal clarification:** In this supplied corpus, the named Act is **Document #9**, not #10. Document #10 is the Local Taxes and Fees Act (Tourist tax section). Select by the exact source subtitle so the example follows the Act you requested rather than the conflicting number.

The four displays show: (1) exact source fields with evidence; (2) generated identity/software/file information; (3) structural decisions and their evidence; (4) deferred fields with no inferred values. Source metadata text is shown in full, including the entire Note.

The structural examples reuse Step 5/6 annotations; no new legal interpretation is performed. Expand the full categorized record for all evidence and candidate annotations. No chunk inheritance policy is selected.

In [ ]:
workflow.step_7d_actual_document_example_black_sea_coast_spatial_development_act()


### 7e. Check metadata fidelity and category separation

**What/how:** Compare all 126 metadata rows across all 21 documents with the unchanged Step 6 descriptor rows and actual table paragraph records. Check title, label, value, cell coordinate and evidence fidelity. Compare package properties directly to XML text/attributes. Confirm deferred entries contain decisions/reasons only, without computed values or chunk inheritance choices.

A before/after digest checks every pre-existing Step 6 field. Small synthetic metadata rows additionally check that unknown/duplicate labels and an empty value remain intact. These are focused checks of Step 7 bookkeeping, not Step 8's full validation/export workflow or evaluation.

**Inspect:** A compact pass summary below. Source “Version date” and “Accessed” remain strings; package timestamps remain separately scoped raw properties.

In [ ]:
workflow.step_7e_check_metadata_fidelity_and_category_separation()


## Step 8 — Validate parsing fidelity and persist Stage 1

**What:** Check the assembled `corpus` against the supplied DOCX and save `data/processed/01_parsed_docx_structure.json` only when required checks pass.

**Why:** Counts alone cannot detect reordered paragraphs, altered text, or metadata detached from its source cells. We compare actual locations and content, while keeping XML-level checks bounded to the structures this file uses.

**How:** Re-read the same archive member to verify its identity. Compare body/cell paragraphs and table order against XML; reconcile boundaries, regions, metadata, IDs, footer fields and marker evidence. Each check produces a named PASS/FAIL row. Failures prevent export. Earlier notebook assertions also stop execution on errors; use **Restart Kernel and Run All** for a complete run.

**Scope of PASS:** Faithful representation of the supplied DOCX's stored text and supported structures. It is not verification of legal correctness, website completeness, Word page rendering or a final legal hierarchy. Repeated source text is preserved; the test rejects accidental duplicate **records/locations**, not legitimate repeated paragraphs. Title/metadata values intentionally repeated in descriptors are checked as evidence-backed copies, not counted as additional body content.

### 8a. Establish the source and validation helpers

The source archive is opened read-only. We hash it before and after persistence. Validation compares original text without trimming or normalization. Small targeted failure cases exercise malformed input, changed text and a missing metadata structure; they do not change the real source.

In [ ]:
workflow.step_8_validate_parsing_fidelity_and_persist_stage_1()


### 8b. Validate identity, order, text and coverage

Every top-level paragraph/table is compared at its original body position. Table rows/cells and their paragraph positions are checked without flattening tables into additional body text. This accounts for 31,055 body paragraphs plus 252 cell paragraphs, 21 tables, and one separate section-property record.

Boundary checks use the title/subtitle/metadata-table pattern and exact region delimiters in the source XML. The corpus title must remain front matter; 21 full-text markers and 19 optional reference-rule markers must be accounted for. All body blocks and nested cell paragraphs must have the expected membership/region.

In [ ]:
# Helpers for this step are imported by the package workflow.


### 8c. Validate metadata, provenance and structural evidence

IDs must be unique at canonical content locations; references may legitimately point to the same ID. Every document's six metadata rows and Step 7's categorized values must agree with their actual table paragraphs. Package properties remain separately scoped.

Legal-marker checks verify labels/spans, full-text membership and representative Article, Chapter, Section, `§`, and Annex evidence. They do not certify that every candidate is a legal provision. Footer fields and section-property records are checked separately from body text.

In [ ]:
workflow.step_8c_validate_metadata_provenance_and_structural_evidence()


### 8d. Report ambiguity and show source previews

Unclassified headings, repeated heading labels and provisional legal markers are **review items**, not automatically parsing failures. Their text and locations remain available. No classification is invented to make the report look complete.

Inspect a representative marker from each requested category, a suffix-bearing article, a long paragraph, and the end of the final source. The last preview deliberately retains website/footer-like text. A source note claiming website boilerplate was removed does not justify removing text ourselves.

Automated fidelity checks cannot replace your review of semantic boundaries or legal meaning. These outputs are previews for that review; PASS below refers only to the stated automated checks.

In [ ]:
workflow.step_8d_report_ambiguity_and_show_source_previews()


### 8e. Persist and reload the validated representation

**What/how:** Write UTF-8 JSON with readable indentation to a temporary file in `data/processed/`, then atomically replace only `01_parsed_docx_structure.json`. This avoids a partially written final file. The original archive, DOCX bytes and existing PDF/TXT artifact are untouched.

A guard checks that the corpus has not changed since validation. Reload the final file and require exact equality with the in-memory `corpus`. Compare the source archive hash before/after writing. On rerun, compare the output hash with any previous artifact; identical bytes demonstrate repeatable output for unchanged inputs/code. A changed artifact is reported as changed, not silently labeled repeatable.

**Trade-off:** Readable JSON is larger than the compressed DOCX because it includes text, run structure, provenance and annotations. It remains a simple local intermediate artifact, not a database. No timestamps are added to the validation report, so unchanged runs can produce identical output.

This is the only cell in the notebook that writes the processed corpus. Rerunning it replaces that derived file after the validation guard passes.

In [ ]:
workflow.step_8e_persist_and_reload_the_validated_representation()


### Final Stage 1 summary

The summary below records the actual automated validation and persistence results. Known limitations and deferred decisions remain visible in the notebook and artifact diagnostics. The 31,076 structural blocks are top-level body records; the 252 table-cell paragraphs remain nested and are counted separately in fidelity checks.

In [ ]:
workflow.final_stage_1_summary()


## Stop — Stage 1 complete; review required before the next stage

The validated Stage 1 representation is saved at `data/processed/01_parsed_docx_structure.json`. Review the PASS/FAIL table, preserved ambiguity report, source previews, and final summary before approving further work.

No cleaning, chunking, embeddings, vector storage, retrieval, RAG, or evaluation was performed. Original source data remains unchanged. No final legal hierarchy or chunk-metadata inheritance policy has been selected. **Do not begin the next stage without your review and approval.**